In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import pandas as pd
import matplotlib.pyplot as plt
import shap
from xgboost import plot_importance
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

df = pd.read_csv('../data/processed/data_clean.csv')
categorical_cols = df.select_dtypes(include=['str']).columns

for col in categorical_cols:
    df[col] = df[col].astype('category').cat.codes

In [2]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Optuna

In [ ]:
import optuna


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
        "n_jobs": -1,
        "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
        "eval_metric": "logloss",
    }

    threshold = trial.suggest_float("threshold", 0.3, 0.7)

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    y_pred = (proba >= threshold).astype(int)
    return f1_score(y_test, y_pred, pos_label=1)

sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=100)

[I 2026-05-03 22:02:08,334] A new study created in memory with name: no-name-45ad871f-ade1-4b33-81b7-fba71403225b
[I 2026-05-03 22:02:09,696] Trial 0 finished with value: 0.6068052930056711 and parameters: {'n_estimators': 325, 'learning_rate': 0.10428600051128875, 'max_depth': 10, 'subsample': 0.9422409071603914, 'colsample_bytree': 0.6784306006597941, 'min_child_weight': 5, 'gamma': 4.337892257945341, 'reg_alpha': 0.4312117764446466, 'reg_lambda': 2.947660056947146, 'threshold': 0.3634450205321219}. Best is trial 0 with value: 0.6068052930056711.
[I 2026-05-03 22:02:10,128] Trial 1 finished with value: 0.6092436974789915 and parameters: {'n_estimators': 787, 'learning_rate': 0.12550095699426778, 'max_depth': 6, 'subsample': 0.9150263730127324, 'colsample_bytree': 0.6016688191688255, 'min_child_weight': 8, 'gamma': 0.9829627814659603, 'reg_alpha': 2.2459195221719543, 'reg_lambda': 4.709152179540261, 'threshold': 0.4256093758185377}. Best is trial 1 with value: 0.6092436974789915.
[I 2

In [ ]:
print("-" * 30)
print("Best Params:", study.best_params)
print("Best F1 Score:", study.best_value)

In [ ]:
best = study.best_params.copy()
threshold = best.pop("threshold")

best_model = XGBClassifier(
    **best,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric="logloss",
)
best_model.fit(X_train, y_train)

proba = best_model.predict_proba(X_test)[:, 1]
y_pred = (proba >= threshold).astype(int)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred, normalize='true',))

# Analisis de resultados

In [ ]:
# === 1) Importancia nativa con nombres reales ===
try:
    feature_names = best_model[:-1].get_feature_names_out()
except Exception:
    feature_names = None

booster = best_model.get_booster()
if feature_names is not None:
    booster.feature_names = list(feature_names)

for tipo in ["weight", "gain", "cover"]:
    imp = booster.get_score(importance_type=tipo)
    df_imp = pd.DataFrame(imp.items(), columns=["feature", tipo]).sort_values(tipo, ascending=False)
    print(f"\nTop 10 por {tipo}:")
    print(df_imp.head(10))

# === 2) Gráfico ===
fig, ax = plt.subplots(figsize=(8, 8))
plot_importance(booster, max_num_features=15, importance_type="gain", ax=ax)
plt.tight_layout()
plt.show()

# === 3) SHAP usando el pipeline completo ===
sample = X_test.sample(min(200, len(X_test)), random_state=42)

explainer = shap.Explainer(best_model.predict_proba, sample)
shap_values = explainer(sample)

# Para clasificación binaria, tomar la clase positiva (churn=1)
shap_values_pos = shap_values[..., 1]

shap.plots.bar(shap_values_pos)
shap.plots.beeswarm(shap_values_pos)
from sklearn.metrics import precision_score, recall_score, f1_score

proba = best_model.predict_proba(X_test)[:, 1]

results = []
thresholds = np.arange(0.25, 0.85, 0.05)
for threshold in thresholds:
    y_pred_threshold = (proba >= threshold).astype(int)
    precision = precision_score(y_test, y_pred_threshold, pos_label=1)
    recall = recall_score(y_test, y_pred_threshold, pos_label=1)
    f1 = f1_score(y_test, y_pred_threshold, pos_label=1)
    results.append((threshold, precision, recall, f1))
    
df_threshold = pd.DataFrame(results, columns=['Threshold', 'Precision', 'Recall', 'F1'])
df_probs = pd.DataFrame({
    'Real': y_test,
    'Probabilidad': best_model.predict_proba(X_test)[:, 1]
})
plt.figure(figsize=(10, 6))
sns.histplot(data=df_probs[df_probs['Real'] == 0], x='Probabilidad', 
             color='skyblue', label='Clase 0 (Real)', kde=True, bins=30, alpha=0.6)

sns.histplot(data=df_probs[df_probs['Real'] == 1], x='Probabilidad', 
             color='orange', label='Clase 1 (Real)', kde=True, bins=30, alpha=0.6)

plt.axvline(x=0.5, color='red', linestyle='--',)

plt.title('Distribución de Probabilidades por Clase Real')
plt.xlabel('Probabilidad de ser Clase 1')
plt.ylabel('Cantidad de casos')
plt.xticks(np.arange(0, 1.05, 0.05), rotation=45)
plt.legend()
plt.show()
plt.figure(figsize=(10, 6))
df_threshold.plot(x='Threshold', y=['Precision', 'Recall', 'F1'], marker='o')
plt.axvline(x=0.5, color='red', linestyle='--',)
plt.xticks(thresholds, rotation=45)
plt.yticks(np.arange(0, 1.1, 0.1))
plt.show()
df_threshold.sort_values('F1', ascending=False)